# OpenAI Function Calling


In [41]:
from IPython.display import display, HTML
display(HTML(
"""
<a target="_blank" href="https://colab.research.google.com/github/pedrodiamel/agents-mini-course/blob/course/books/aula_04_openai_functions.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
"""
))

In [6]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [42]:
import requests

city = "Recife"
unit = "celsius"
WEATHER_API_KEY = os.environ['WEATHER_API_KEY']
response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units={unit}")
data = response.json()
data

{'coord': {'lon': -34.8811, 'lat': -8.0539},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'light rain',
   'icon': '10d'}],
 'base': 'stations',
 'main': {'temp': 302.17,
  'feels_like': 306.69,
  'temp_min': 302.17,
  'temp_max': 302.17,
  'pressure': 1014,
  'humidity': 74,
  'sea_level': 1014,
  'grnd_level': 1011},
 'visibility': 10000,
 'wind': {'speed': 2.57, 'deg': 70},
 'rain': {'1h': 0.21},
 'clouds': {'all': 40},
 'dt': 1770573130,
 'sys': {'type': 1,
  'id': 8426,
  'country': 'BR',
  'sunrise': 1770538895,
  'sunset': 1770583554},
 'timezone': -10800,
 'id': 3390760,
 'name': 'Recife',
 'cod': 200}

In [13]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""

    # Chama API

    # # Dummy response
    # weather_info = {
    #     "location": location,
    #     "temperature": "72",
    #     "unit": unit,
    #     "forecast": ["sunny", "windy"],
    # }

    units = {"fahrenheit": "imperial", "celsius": "metric", "kelvin": "standard"}
    if unit not in units:
        raise ValueError("Invalid unit. Must be 'fahrenheit', 'celsius', or 'kelvin'.")

    response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid={WEATHER_API_KEY}&units={units[unit]}")
    data = response.json()
    weather_info = {
        "location": location,
        "temperature": data['main']['temp'],
        "unit": unit,
        "forecast": [data['weather'][0]['description']],
    }

    return json.dumps(weather_info)

In [14]:
get_current_weather("Recife", "celsius")

'{"location": "Recife", "temperature": 29.02, "unit": "celsius", "forecast": ["moderate rain"]}'

In [15]:
# define a function
# https://platform.openai.com/docs/guides/function-calling

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        }
    }
]

In [16]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Recife?"
    }
]

In [17]:
import openai

llm_model = "gpt-4o-mini"
client = openai.OpenAI()


In [18]:
# Call the responses endpoint

response = client.chat.completions.create(
        model=llm_model,
        messages=messages,
        tools=tools,
        temperature=0,
    )

In [19]:
print(response)

ChatCompletion(id='chatcmpl-D7369ZbRv5PHDD83Pwyj9cgtNzUHB', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7jJKTQm7AEApSqORAn0NT7F0', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')]))], created=1770572577, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_f4ae844694', usage=CompletionUsage(completion_tokens=18, prompt_tokens=79, total_tokens=97, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [20]:
response_message = response.choices[0].message

In [21]:
response_message

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_7jJKTQm7AEApSqORAn0NT7F0', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')])

In [22]:
print(response_message.content)

None


In [23]:
print(response_message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_7jJKTQm7AEApSqORAn0NT7F0', function=Function(arguments='{"location":"Recife, Brazil"}', name='get_current_weather'), type='function')]


In [24]:
#for tool_call in response_message.tool_calls:
#    print("Tool Call Name:", tool_call.function.name)
#    print("Tool Call Arguments:", json.loads(tool_call.function.arguments))

print("Function Name:", response_message.tool_calls[0].function.name)
print("Function Arguments:", json.loads(response_message.tool_calls[0].function.arguments))

Function Name: get_current_weather
Function Arguments: {'location': 'Recife, Brazil'}


In [25]:
args = json.loads(response_message.tool_calls[0].function.arguments)
args

{'location': 'Recife, Brazil'}

In [28]:
get_current_weather(**args)

'{"location": "Recife, Brazil", "temperature": 84.24, "unit": "fahrenheit", "forecast": ["light rain"]}'

### LangChain Tools

In [30]:
from langchain.tools import tool
import requests

In [32]:
@tool
def get_weather(city: str, unit: str = "celsius") -> str:
    """
    Obtém a previsão do tempo para a cidade informada.
    Args:
        city: Nome da cidade (ex: 'Recife, PE, Brasil')
        unit: Unidade de temperatura ('fahrenheit', 'celsius', 'kelvin')
    Returns:
        String com resumo da previsão do tempo.
    """

    # Chama API

    # Dummy response
    # resumo = f"[Serviço fictício] A previsão para {city} é: 26°C, céu parcialmente nublado."

    # Real call
    units = {"fahrenheit": "imperial", "celsius": "metric", "kelvin": "standard"}
    if unit not in units:
        raise ValueError("Invalid unit. Must be 'fahrenheit', 'celsius', or 'kelvin'.")

    response = requests.get(f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units={units[unit]}")
    data = response.json()
    weather_info = {
        "location": city,
        "temperature": data['main']['temp'],
        "unit": unit,
        "forecast": [data['weather'][0]['description']],
    }

    resumo = f"A temperatura em {city} agora é {weather_info['temperature']}°{unit[0].upper()} com {', '.join(weather_info['forecast'])}."
    return resumo


@tool
def thinking_tool(reflexion:str):
    """Chame a ferramenta de reflexão para analisar a resposta."""
    return f"[Ferramenta de reflexão] Analisando: {reflexion}"


In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=llm_model,
    temperature=0.7
    )

# Vincula a tool ao modelo
llm_with_tools = llm.bind_tools([thinking_tool, get_weather])

In [39]:
from langchain_core.messages import (
    HumanMessage,
    ToolMessage,
)


user_input = "Qual a previsão do tempo para Recife hoje em celsius?"
result = llm_with_tools.invoke([HumanMessage(content=user_input)])


# Verifica se houve chamado da tool
if hasattr(result, "tool_calls") and result.tool_calls:

    tool_call = result.tool_calls[0]
    tool_name = tool_call["name"]
    args = tool_call["args"]


    if tool_name != "get_weather":
        raise ValueError(f"Tool inesperada: {tool_name}")

    # Executa a tool
    tool_result = get_weather.invoke(args)

    tool_message = ToolMessage(
        name=tool_name,
        content=str(tool_result),
        tool_call_id=tool_call["id"]
    )


    # Envia de volta ao modelo o resultado da tool, para que ele complete a resposta
    followup = llm_with_tools.invoke([
        HumanMessage(content=user_input),
        result,
        tool_message
    ])

    print(followup.content)

else:
    print(result.content)

A temperatura em Recife, PE, Brasil, agora é de 29,25°C, com nuvens fragmentadas.


In [40]:
result.tool_calls[0]

{'name': 'get_weather',
 'args': {'city': 'Recife, PE, Brasil', 'unit': 'celsius'},
 'id': 'call_wjuwK4K7oVD46axbPY3NBlZK',
 'type': 'tool_call'}